In [1]:
import pandas as pd
import numpy as np

In [2]:
#loading dataset
df = pd.read_parquet("../outputs/pheme_tweets.parquet")
print("Original shape: ",df.shape)
print("threads:", df["thread_id"].nunique())
print("events:", df["event"].nunique())
df.head()

Original shape:  (103212, 99)
threads: 5802
events: 5


,contributors,truncated,text,in_reply_to_status_id,id,favorite_count,source,retweeted,coordinates,in_reply_to_screen_name,...,user_profile_location,possibly_sensitive_appealable,metadata_iso_language_code,metadata_result_type,place_attributes_street_address,n_hashtags,n_urls,n_mentions,n_media,n_symbols
0,None,False,EXTENDED: Dramatic video of gunfire inside hal...,NaN,524950899899113473,132,"<a href=""https://about.twitter.com/products/tw...",False,NaN,NaN,...,NaN,None,NaN,NaN,NaN,0,1,0,1,0
1,None,False,“@ctvottawa: Gunfire inside hallways of Parlia...,5.249509e+17,524966168772485120,0,"<a href=""http://twitter.com/download/iphone"" r...",False,NaN,ctvottawa,...,NaN,None,NaN,NaN,NaN,0,0,2,0,0
2,None,False,@ctvottawa so proud of our parliament security...,5.249509e+17,524957878768394240,1,"<a href=""http://twitter.com/download/iphone"" r...",False,NaN,ctvottawa,...,NaN,None,NaN,NaN,NaN,0,0,1,0,0
3,None,False,@ctvottawa @DerianPlouffe,5.249509e+17,524954849637847040,0,"<a href=""http://twitter.com/download/iphone"" r...",False,NaN,ctvottawa,...,NaN,None,NaN,NaN,NaN,0,0,2,0,0
4,None,False,@ctvottawa two attacks on CAF personnel in two...,5.249509e+17,525005502468849664,0,"<a href=""http://twitter.com/download/android"" ...",False,NaN,ctvottawa,...,NaN,None,NaN,NaN,NaN,0,0,1,0,0


In [3]:
# Basic column check
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

Columns:
['contributors', 'truncated', 'text', 'in_reply_to_status_id', 'id', 'favorite_count', 'source', 'retweeted', 'coordinates', 'in_reply_to_screen_name', 'id_str', 'retweet_count', 'in_reply_to_user_id', 'favorited', 'geo', 'in_reply_to_user_id_str', 'possibly_sensitive', 'lang', 'created_at', 'in_reply_to_status_id_str', 'place', 'event', 'label', 'label_name', 'thread_id', 'is_source_tweet', 'entities_symbols', 'entities_user_mentions', 'entities_hashtags', 'entities_urls', 'entities_media', 'user_follow_request_sent', 'user_profile_use_background_image', 'user_default_profile_image', 'user_id', 'user_profile_background_image_url_https', 'user_verified', 'user_profile_text_color', 'user_profile_image_url_https', 'user_profile_sidebar_fill_color', 'user_entities_url_urls', 'user_entities_description_urls', 'user_followers_count', 'user_profile_sidebar_border_color', 'user_id_str', 'user_profile_background_color', 'user_listed_count', 'user_is_translation_enabled', 'user_utc_off

In [4]:
#making a copy and only keeping engish language
df_en = df[df["lang"]=="en"].copy()
print("After English filtering:",df_en.shape)
print("Unique threads:", df_en["thread_id"].nunique())
print("Unique events:", df_en["event"].nunique())

After English filtering: (95167, 99)
Unique threads: 5802
Unique events: 5


In [5]:
#Convert Timestamp
df_en["created_at"] = pd.to_datetime(
    df_en["created_at"],
    errors="coerce"
)

print("Invalid timestamps:", df_en["created_at"].isna().sum())

/var/folders/k3/4xltyhr53vdgsxsvpd_vkhkw0000gn/T/ipykernel_67521/3657720778.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_en["created_at"] = pd.to_datetime(


Invalid timestamps: 0


In [6]:
# Dropping rows with missing core values
core_cols = [
    "id",
    "thread_id",
    "user_id",
    "event",
    "label_name",
    "text",
    "created_at"
]
df_en = df_en.dropna(subset=core_cols).copy()

print("After dropping missing core rows:", df_en.shape)

After dropping missing core rows: (95167, 99)


In [7]:
#Converting ID columns to string

id_cols = ["id", "thread_id", "user_id"]

for col in id_cols:
    if col in df_en.columns:
        df_en[col] = df_en[col].astype(str)

df_en[id_cols].dtypes

id           str
thread_id    str
user_id      str
dtype: object

In [8]:
df_en["id"].duplicated().sum()

np.int64(772)

In [9]:
dups = df_en[df_en["id"].duplicated(keep=False)]

dups[["id", "thread_id", "text"]].sort_values("id").head(20)

,id,thread_id,text
88378,498235547685756928,498235547685756928,Black teenage boys are not men. They are child...
88383,498235547685756928,498235547685756928,Black teenage boys are not men. They are child...
95054,498248648699150336,498248648699150336,Police have brought out the large gear in #Fer...
95110,498248648699150336,498248648699150336,Police have brought out the large gear in #Fer...
96753,498251940997136384,498251940997136384,When you see citizens as enemies... RT @Antoni...
96754,498251940997136384,498251940997136384,When you see citizens as enemies... RT @Antoni...
86913,498253149707464704,498253149707464704,Police of #Ferguson murdered a boy in cold blo...
86915,498253149707464704,498253149707464704,Police of #Ferguson murdered a boy in cold blo...
102924,498253652755111937,498253652755111937,Here’s to hoping what’s happening in STL right...
102931,498253652755111937,498253652755111937,Here’s to hoping what’s happening in STL right...


In [10]:
#Removing duplicated tweet IDs
before = len(df_en)
df_en = (df_en
         .sort_values("is_source_tweet", ascending=False)
         .drop_duplicates(subset="id", keep="first")
         .reset_index(drop=True))
print(f"Removed {before - len(df_en)} duplicate rows")
print("After dedup shape:", df_en.shape)
print("Duplicate ids remaining:", df_en["id"].duplicated().sum())
print("Unique threads:", df_en["thread_id"].nunique())
print("Unique events:", df_en["event"].nunique())

Removed 772 duplicate rows
After dedup shape: (94395, 99)
Duplicate ids remaining: 0
Unique threads: 5802
Unique events: 5


In [11]:
#Sort tweets within each thread by time
df_en = df_en.sort_values(
    by=["thread_id", "created_at"]
).reset_index(drop=True)

df_en.head()

,contributors,truncated,text,in_reply_to_status_id,id,favorite_count,source,retweeted,coordinates,in_reply_to_screen_name,...,user_profile_location,possibly_sensitive_appealable,metadata_iso_language_code,metadata_result_type,place_attributes_street_address,n_hashtags,n_urls,n_mentions,n_media,n_symbols
0,None,False,Black teenage boys are not men. They are child...,NaN,498235547685756928,124,"<a href=""http://twitter.com/#!/download/ipad"" ...",False,NaN,NaN,...,NaN,None,NaN,NaN,NaN,1,0,0,0,0
1,None,False,@annaxsweat @NeoSoulPol Same thing when #Trayv...,4.982355e+17,498243332204949504,0,"<a href=""https://about.twitter.com/products/tw...",False,NaN,annaxsweat,...,NaN,None,NaN,NaN,NaN,3,0,2,0,0
2,None,False,@annaxsweat #StopThugCops We have to take acti...,4.982355e+17,498266827676741632,2,"<a href=""http://twitter.com/#!/download/ipad"" ...",False,NaN,annaxsweat,...,NaN,None,NaN,NaN,NaN,2,0,1,0,0
3,None,False,@annaxsweat @KidFriendlyCamb 17 year olds can ...,4.982355e+17,498272808560889858,0,"<a href=""http://twitter.com"" rel=""nofollow"">Tw...",False,NaN,annaxsweat,...,NaN,None,NaN,NaN,NaN,0,0,2,0,0
4,None,False,"@annaxsweat well, he damn sure ain't a child",4.982355e+17,498312902928244736,1,"<a href=""http://twitter.com"" rel=""nofollow"">Tw...",False,NaN,annaxsweat,...,NaN,None,NaN,NaN,NaN,0,0,1,0,0


In [12]:
df_en[["thread_id", "id", "created_at", "text"]].head(10)

,thread_id,id,created_at,text
0,498235547685756928,498235547685756928,2014-08-09 22:33:06+00:00,Black teenage boys are not men. They are child...
1,498235547685756928,498243332204949504,2014-08-09 23:04:01+00:00,@annaxsweat @NeoSoulPol Same thing when #Trayv...
2,498235547685756928,498266827676741632,2014-08-10 00:37:23+00:00,@annaxsweat #StopThugCops We have to take acti...
3,498235547685756928,498272808560889858,2014-08-10 01:01:09+00:00,@annaxsweat @KidFriendlyCamb 17 year olds can ...
4,498235547685756928,498312902928244736,2014-08-10 03:40:28+00:00,"@annaxsweat well, he damn sure ain't a child"
5,498235547685756928,498313376888799232,2014-08-10 03:42:21+00:00,@HighPlainsCoot fuck. Off.
6,498235547685756928,498314073873055744,2014-08-10 03:45:08+00:00,@annaxsweat the response of a leftist toker. i...
7,498235547685756928,498411979091218432,2014-08-10 10:14:10+00:00,@annaxsweat such talk from a lady(?). senseles...
8,498235547685756928,498519407375941632,2014-08-10 17:21:03+00:00,@annaxsweat: Soooo wonderfully said! Thank you...
9,498235547685756928,498785502062190592,2014-08-11 10:58:25+00:00,@AlexisAStevens yeah that tweet was written be...


In [13]:
#Identify source tweet time for each thread
source_times = (
    df_en[df_en["is_source_tweet"] == True]
    .groupby("thread_id")["created_at"]
    .min()
    .rename("source_time")
)

df_en = df_en.merge(
    source_times,
    on="thread_id",
    how="left"
)

df_en[["thread_id", "created_at", "source_time"]].head()

# created_at = when that specific tweet was posted.
# source_time = when the thread started (the source tweet's timestamp).

,thread_id,created_at,source_time
0,498235547685756928,2014-08-09 22:33:06+00:00,2014-08-09 22:33:06+00:00
1,498235547685756928,2014-08-09 23:04:01+00:00,2014-08-09 22:33:06+00:00
2,498235547685756928,2014-08-10 00:37:23+00:00,2014-08-09 22:33:06+00:00
3,498235547685756928,2014-08-10 01:01:09+00:00,2014-08-09 22:33:06+00:00
4,498235547685756928,2014-08-10 03:40:28+00:00,2014-08-09 22:33:06+00:00


In [14]:
#Calculate time since source tweet
df_en["time_since_source_min"] = (
    df_en["created_at"] - df_en["source_time"]
).dt.total_seconds() / 60

df_en["time_since_source_min"].describe()

count    94395.000000
mean       291.958991
std       1128.106320
min          0.000000
25%          6.450000
50%         28.283333
75%        120.341667
max      43725.366667
Name: time_since_source_min, dtype: float64

In [15]:
#Check temporal validity
invalid_time = df_en[df_en["time_since_source_min"] < 0]

print("Tweets posted before source tweet:", len(invalid_time))

Tweets posted before source tweet: 0


In [16]:
#Check threads with missing source tweet
missing_source_threads = df_en[
    df_en["source_time"].isna()
]["thread_id"].nunique()

print("Threads with missing source tweet:", missing_source_threads)

Threads with missing source tweet: 0


In [17]:
# Final prepared dataset summary
summary = {
    "tweets": len(df_en),
    "threads": df_en["thread_id"].nunique(),
    "events": df_en["event"].nunique(),
    "users": df_en["user_id"].nunique() if "user_id" in df_en.columns else np.nan,
    "rumours": (df_en["label_name"] == "rumours").sum(),
    "non_rumours": (df_en["label_name"] == "non-rumours").sum()
}

pd.DataFrame([summary])

,tweets,threads,events,users,rumours,non_rumours
0,94395,5802,5,45154,28680,65715


In [19]:
#Event x Label
event_label_en = pd.crosstab(
    df_en["event"],
    df_en["label_name"]
)
event_label_en

label_name,non-rumours,rumours
event,,
charliehebdo,27895,6543
ferguson,16388,6077
germanwings-crash,1685,2222
ottawashooting,5425,5920
sydneysiege,14322,7918


In [18]:
#Save prepared dataset

df_en.to_parquet("../outputs/pheme_prepared_english.parquet", index=False)

print("Saved")

Saved
